<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/OB_0911_%EC%95%99%EC%83%81%EB%B8%94_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **앙상블 학습과 랜덤 포레스트 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch07 앙상블 학습과 랜덤 포레스트 연습문제 2, 7, 8, 9번
- 이론적 지식을 묻는 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

In [1]:
# import libraries
import numpy as np

### **1. 직접 투표와 간접 투표 분류기 사이의 차이점은 무엇일까요?**
___


직접 투표 분류기 (Hard Voting) : 앙상블에 포함된 각 개별 분류기가 예측한 '클래스(Class)'를 모아서 가장 많은 표를 얻은 클래스를 최종 예측으로 선택한다.
- 다수결의 원칙을 따른다

간접 투표 분류기 (Soft Voting): 앙상블에 포함된 각 개별 분류기가 예측한 '클래스별 확률'을 모두 더한 뒤 평균을 낸다. 그중 가장 높은 평균 확률을 가진 클래스를 최종 예측으로 선택한다.
- 확률이 높은 분류기의 예측에 더 많은 가중치를 줌

### **2. 그레디언트 부스팅 앙상블이 훈련 데이터에 과대 적합되었다면 학습률을 어떻게 해야 할까요?**
___

학습률(learning rate)을 낮춰야 한다.

-> 학습률을 감소시키면 모델에 추가되는 각 트리의 기여도가 줄어든다. 즉, 모델이 훈련 데이터의 아주 세세한 패턴이나 노이즈까지 급격하게 학습하는 것을 방지하여 강력한 규제(Regularization) 효과를 얻을 수 있다.

### **3. [실습] 다음 지시에 따라 투표 기반 분류 모델을 만들어 보세요**
___

#### **STEP 1. MNIST 데이터를 불러들이고, 훈련, 검증, 테스트 데이터로 나누세요.**

In [2]:
# import MNIST dataset
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame = False)
X, y = mnist["data"], mnist["target"]

In [3]:
# train/valid/test dataset
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=10000, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=10000, random_state=42)

####  **STEP 2. 랜덤 포레스트 분류기, 엑스트라 트리 분류기, SVM 분류기, MLP 분류기를 훈련시키세요.**
- 모델 파라미터는 `n_estimators=100`, `random_state=42`로 설정합니다.

In [4]:
# import package
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier

In [5]:
# model fitting
random_forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
extra_trees_clf = ExtraTreesClassifier(n_estimators=100, random_state=42)
svm_clf = LinearSVC(max_iter=100, tol=20, random_state=42)
mlp_clf = MLPClassifier(random_state=42)

estimators = [random_forest_clf, extra_trees_clf, svm_clf, mlp_clf]

for estimator in estimators:
    print("Training the", estimator)
    estimator.fit(X_train, y_train)

Training the RandomForestClassifier(random_state=42)
Training the ExtraTreesClassifier(random_state=42)
Training the LinearSVC(max_iter=100, random_state=42, tol=20)
Training the MLPClassifier(random_state=42)


####  **STEP 3-1. 앞에서 훈련시킨 각 모델을 직접 투표 방법을 사용해 앙상블로 연결하고 훈련시킨 후, `score()`메서드를 이용하여 검증 데이터셋에서의 성능을 평가해보세요.**

In [32]:
from sklearn.ensemble import VotingClassifier

# 앙상블에 사용할 모델들을 (이름, 모델객체) 형태의 튜플 리스트로 묶어줌
named_estimators = [
    ("rf_clf", random_forest_clf),
    ("ext_clf", extra_trees_clf),
    ("svm_clf", svm_clf),
    ("mlp_clf", mlp_clf)
]

# 직접 투표 앙상블 모델 생성 (voting='hard')
voting_clf = VotingClassifier(estimators=named_estimators, voting='hard')

In [ ]:
# model fitting
voting_clf.fit(X_train, y_train)

In [ ]:
# model test
voting_score = voting_clf.score(X_val, y_val)
print(f"직접 투표 분류기(Hard Voting) 검증 세트 정확도: {voting_score:.4f}")

####  **STEP 3-2. 검증 데이터셋에서 각 분류 모델의 성능을 `score()` 메서드를 이용하여 확인해보고, 가장 성능이 낮은 모델을 제거하여 그 결과를 비교해보세요.**
- Hint : 가장 성능이 낮은 모델을 제거할 때 `del`를 활용해보세요

In [ ]:
# 각 분류 모델 학습
voting_clf.estimators_

In [ ]:
# 각 분류 모델의 성능 확인
[estimator.score(X_val, y_val) for estimator in estimators]

- Q. 어떤 모델의 성능이 가장 낮나요?
- A. SVM 분류 모델

In [ ]:
# 가장 성능이 낮은 모델 제거
del voting_clf.estimators[2]

In [ ]:
# 모델 제거 후 성능 확인
voting_clf.score(X_val, y_val)

### **4. 다음 단계를 따라 앞에서 훈련시킨 분류 모델들을 이용하여 스태킹 앙상블을 구성해보자.**
___

#### **STEP 1. 3번 문제의 각 분류 모델을 실행해서 검증 세트에서 예측을 만들고, 그 결과로 훈련 세트를 만들어 보세요.**

In [ ]:
# 새 훈련 세트를 저장할 ndarray 생성
X_val_predictions = np.empty((len(X_val), len(estimators)), dtype=np.float32)

####  **STEP 2. 새로운 훈련 세트를 이용하여 랜덤 포레스트 분류 모델을 학습시켜 보세요.**

In [ ]:
rnd_forest_blender = RandomForestClassifier(n_estimators=200, random_state=42)
rnd_forest_blender.fit(X_val_predictions, y_val)

- 이 랜덤 포레스트 분류 모델이 바로 블렌더에 해당합니다.

####  **STEP 3. 이제 테스트셋에서 스태킹 앙상블 모델을 평가해보세요.**
- 성능 평가 지표로 **정확도**를 이용하세요.

In [ ]:
# 각 분류 모델의 예측을 만들어 새로운 데이터셋 생성
X_test_predictions = np.empty((len(X_test), len(estimators)), dtype=np.float32)

for index, estimator in enumerate(estimators):
    X_test_predictions[:, index] = estimator.predict(X_test)

In [ ]:
# 새로운 데이터셋을 이용하여 블렌더로 예측
y_pred = rnd_forest_blender.predict(X_test_predictions)

In [ ]:
# model test
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)